In [1]:
# ============================================================
# TOPIC MODELING PIPELINE (LDA, LSA, NMF)
# ============================================================

# -----------------------------
# STEP 0: Install libraries
# -----------------------------
!pip install nltk gensim scikit-learn

# ============================================================
# STEP 1: USER PARAMETERS (EDITABLE)
# ============================================================
NUM_TOPICS = 5        # Number of topics
NUM_WORDS = 10        # Top words per topic
LDA_PASSES = 10       # Iterations
MAX_FEATURES = 500    # Vocabulary size

# -----------------------------
# STEP 2: Upload dataset
# -----------------------------
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# -----------------------------
# STEP 3: Load dataset
# -----------------------------
import pandas as pd

df = pd.read_csv(file_name, encoding='latin-1')

# Clean column names
df.columns = df.columns.str.strip().str.lower()

# Auto-detect text column
possible_cols = ['uaqteresponse', 'response', 'text', 'comments']
text_col = next((col for col in possible_cols if col in df.columns), None)

if text_col is None:
    print(df.columns)
    raise ValueError("No valid text column found")

print(f"\nUsing column: {text_col}")

texts = df[text_col].fillna('').astype(str)
texts = texts[texts.str.strip() != '']

# -----------------------------
# STEP 4: Preprocessing
# -----------------------------
import nltk, re
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return tokens

processed_texts = texts.apply(preprocess)

# -----------------------------
# STEP 5: LDA MODEL
# -----------------------------
from gensim.corpora import Dictionary
from gensim.models import LdaModel

dictionary = Dictionary(processed_texts)
corpus = [dictionary.doc2bow(text) for text in processed_texts]

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    passes=LDA_PASSES,
    random_state=42
)

print("\n========== LDA TOPICS ==========")
for i, topic in lda_model.print_topics(num_words=NUM_WORDS):
    print(f"Topic {i}: {topic}")

# -----------------------------
# STEP 6: TF-IDF (for LSA & NMF)
# -----------------------------
from sklearn.feature_extraction.text import TfidfVectorizer

texts_joined = processed_texts.apply(lambda x: " ".join(x))

vectorizer = TfidfVectorizer(max_features=MAX_FEATURES)
X = vectorizer.fit_transform(texts_joined)
terms = vectorizer.get_feature_names_out()

# -----------------------------
# STEP 7: LSA MODEL
# -----------------------------
from sklearn.decomposition import TruncatedSVD

lsa_model = TruncatedSVD(n_components=NUM_TOPICS, random_state=42)
X_lsa = lsa_model.fit_transform(X)

print("\n========== LSA TOPICS ==========")
for i, comp in enumerate(lsa_model.components_):
    words = [terms[j] for j in comp.argsort()[::-1][:NUM_WORDS]]
    print(f"Topic {i}: {words}")

# -----------------------------
# STEP 8: NMF MODEL
# -----------------------------
from sklearn.decomposition import NMF

nmf_model = NMF(n_components=NUM_TOPICS, random_state=42)
X_nmf = nmf_model.fit_transform(X)

print("\n========== NMF TOPICS ==========")
for i, comp in enumerate(nmf_model.components_):
    words = [terms[j] for j in comp.argsort()[::-1][:NUM_WORDS]]
    print(f"Topic {i}: {words}")

# ============================================================
# STEP 9: EVALUATION
# ============================================================
from gensim.models import CoherenceModel
import numpy as np
from sklearn.metrics import mean_squared_error

def extract_topics(model, terms):
    topics = []
    for comp in model.components_:
        words = [terms[i] for i in comp.argsort()[::-1][:NUM_WORDS]]
        topics.append(words)
    return topics

lsa_topics = extract_topics(lsa_model, terms)
nmf_topics = extract_topics(nmf_model, terms)

# LDA coherence
coh_lda = CoherenceModel(
    model=lda_model,
    texts=processed_texts,
    dictionary=dictionary,
    coherence='c_v'
).get_coherence()

# LSA coherence
coh_lsa = CoherenceModel(
    topics=lsa_topics,
    texts=processed_texts,
    dictionary=dictionary,
    coherence='c_v'
).get_coherence()

# NMF coherence
coh_nmf = CoherenceModel(
    topics=nmf_topics,
    texts=processed_texts,
    dictionary=dictionary,
    coherence='c_v'
).get_coherence()

print("\n========== COHERENCE SCORES ==========")
print(f"LDA: {round(coh_lda,3)}")
print(f"LSA: {round(coh_lsa,3)}")
print(f"NMF: {round(coh_nmf,3)}")

# Perplexity (LDA only)
print("\nPerplexity (LDA):", round(lda_model.log_perplexity(corpus), 3))

# Reconstruction Error
lsa_recon = np.dot(X_lsa, lsa_model.components_)
nmf_recon = np.dot(X_nmf, nmf_model.components_)

lsa_err = mean_squared_error(X.toarray(), lsa_recon)
nmf_err = mean_squared_error(X.toarray(), nmf_recon)

print("\nReconstruction Error:")
print(f"LSA: {round(lsa_err,3)}")
print(f"NMF: {round(nmf_err,3)}")

# ============================================================
# INTERPRETATION GUIDE
# ============================================================
print("\n================ INTERPRETATION GUIDE ================")
print("""
COHERENCE SCORE:
- > 0.5 → Good topics
- > 0.7 → Very strong topics
- < 0.4 → Weak topics

PERPLEXITY (LDA):
- Lower = better statistical fit
- Does NOT guarantee interpretability

RECONSTRUCTION ERROR:
- Lower = better mathematical fit
- Does NOT ensure meaningful topics

KEY INSIGHT:
Best model = High coherence + interpretable topics

TUNING TIPS:
- Increase NUM_TOPICS → more specific topics
- Decrease NUM_TOPICS → broader topics
- Increase LDA_PASSES → more stable topics
- Adjust MAX_FEATURES → reduce noise

REMEMBER:
Topic modeling is BOTH mathematical AND interpretative.
""")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 37.5 MB/s eta 0:00:00


Saving UAQTEresponses.csv to UAQTEresponses.csv

Using column: response


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.



========== LDA TOPICS ==========
Topic 0: 0.023*"help" + 0.022*"uaqte" + 0.018*"education" + 0.015*"helped" + 0.014*"students" + 0.013*"university" + 0.012*"able" + 0.011*"family" + 0.011*"free" + 0.010*"tuition"
Topic 1: 0.024*"free" + 0.023*"uaqte" + 0.022*"education" + 0.021*"tuition" + 0.019*"one" + 0.018*"family" + 0.014*"lot" + 0.014*"help" + 0.013*"students" + 0.012*"also"
Topic 2: 0.037*"education" + 0.022*"financial" + 0.019*"uaqte" + 0.017*"quality" + 0.017*"beneficiaries" + 0.016*"access" + 0.014*"one" + 0.012*"life" + 0.012*"college" + 0.011*"tuition"
Topic 3: 0.022*"tuition" + 0.022*"school" + 0.020*"expenses" + 0.019*"education" + 0.016*"uaqte" + 0.016*"free" + 0.014*"help" + 0.014*"without" + 0.013*"good" + 0.013*"study"
Topic 4: 0.026*"uaqte" + 0.018*"one" + 0.014*"education" + 0.013*"helps" + 0.013*"financial" + 0.012*"family" + 0.012*"tuition" + 0.012*"pursue" + 0.012*"im" + 0.011*"beneficiary"

========== LSA TOPICS ==========
Topic 0: ['uaqte', 'education', 'free',